# HMM tipping analysis (standalone)

Fits a 2-state Gaussian HMM over the per-token surprise sequences from the
draft-surprise pipeline (normal vs tipped mode; transitions = tipping
dynamics; baseline as control). **CPU runtime is enough.** Needs the score
files on Drive under `MyDrive/weirdspec/data/`. Run both cells top to bottom.

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Phase 6 — latent-state "tipping" analysis via a 2-state Gaussian HMM
# (self-contained copy of phase6_hmm.py; CPU runtime is enough, runs in minutes)

import json
from collections import defaultdict
from typing import Any

def iter_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def excess_sequence(row: dict[str, Any]):
    import numpy as np

    return np.asarray(row["nll_draft"], dtype=float) - np.asarray(
        row["nll_target"], dtype=float
    )


def _logsumexp(a, axis):
    import numpy as np

    m = a.max(axis=axis, keepdims=True)
    return (m + np.log(np.exp(a - m).sum(axis=axis, keepdims=True))).squeeze(axis)


def fit_gaussian_hmm(sequences, k: int = 2, iters: int = 100, tol: float = 1e-5):
    """Baum-Welch for a 1-D Gaussian HMM, shared over all sequences.

    Deterministic quantile initialization; returns dict with pi, A, means,
    vars, loglik. States are sorted by mean ascending (state 0 = calmest).
    """
    import numpy as np

    x_all = np.concatenate(sequences)
    means = np.quantile(x_all, [(i + 0.5) / k for i in range(k)]).astype(float)
    variances = np.full(k, max(float(x_all.var()), 1e-3))
    pi = np.full(k, 1.0 / k)
    A = np.full((k, k), 0.05 / max(k - 1, 1))
    np.fill_diagonal(A, 0.95)

    prev_ll = -np.inf
    for _ in range(iters):
        log_pi, log_A = np.log(pi), np.log(A)
        ll_total = 0.0
        gamma0_sum = np.zeros(k)
        xi_sum = np.zeros((k, k))
        gamma_notlast_sum = np.zeros(k)
        w_sum = np.zeros(k)
        wx_sum = np.zeros(k)
        wx2_sum = np.zeros(k)

        for x in sequences:
            T = len(x)
            logB = -0.5 * (
                np.log(2 * np.pi * variances)[None, :]
                + (x[:, None] - means[None, :]) ** 2 / variances[None, :]
            )
            la = np.empty((T, k))
            la[0] = log_pi + logB[0]
            for t in range(1, T):
                la[t] = logB[t] + _logsumexp(la[t - 1][:, None] + log_A, axis=0)
            lb = np.zeros((T, k))
            for t in range(T - 2, -1, -1):
                lb[t] = _logsumexp(log_A + (logB[t + 1] + lb[t + 1])[None, :], axis=1)
            last = la[-1]
            ll = float(last.max() + np.log(np.exp(last - last.max()).sum()))
            ll_total += ll
            gamma = np.exp(la + lb - ll)
            gamma0_sum += gamma[0]
            for t in range(T - 1):
                xi = np.exp(
                    la[t][:, None] + log_A + (logB[t + 1] + lb[t + 1])[None, :] - ll
                )
                xi_sum += xi
            gamma_notlast_sum += gamma[:-1].sum(axis=0)
            w_sum += gamma.sum(axis=0)
            wx_sum += (gamma * x[:, None]).sum(axis=0)
            wx2_sum += (gamma * (x[:, None] ** 2)).sum(axis=0)

        pi = gamma0_sum / len(sequences)
        A = xi_sum / np.maximum(gamma_notlast_sum[:, None], 1e-12)
        A /= A.sum(axis=1, keepdims=True)
        means = wx_sum / np.maximum(w_sum, 1e-12)
        variances = np.maximum(wx2_sum / np.maximum(w_sum, 1e-12) - means**2, 1e-4)
        if abs(ll_total - prev_ll) < tol * max(abs(prev_ll), 1.0):
            prev_ll = ll_total
            break
        prev_ll = ll_total

    order = np.argsort(means)
    return {
        "pi": pi[order],
        "A": A[np.ix_(order, order)],
        "means": means[order],
        "vars": variances[order],
        "loglik": float(prev_ll),
    }


def viterbi(x, model):
    """Most likely state path for one sequence under the fitted model."""
    import numpy as np

    means, variances = model["means"], model["vars"]
    log_pi, log_A = np.log(model["pi"] + 1e-12), np.log(model["A"] + 1e-12)
    logB = -0.5 * (
        np.log(2 * np.pi * variances)[None, :]
        + (x[:, None] - means[None, :]) ** 2 / variances[None, :]
    )
    T, k = logB.shape
    delta = np.empty((T, k))
    back = np.zeros((T, k), dtype=int)
    delta[0] = log_pi + logB[0]
    for t in range(1, T):
        scores = delta[t - 1][:, None] + log_A
        back[t] = scores.argmax(axis=0)
        delta[t] = scores.max(axis=0) + logB[t]
    path = np.empty(T, dtype=int)
    path[-1] = int(delta[-1].argmax())
    for t in range(T - 2, -1, -1):
        path[t] = back[t + 1][path[t + 1]]
    return path


def segment_stats(path, top_state: int) -> dict[str, Any]:
    """First entry into the top state, and how much of the sequence it holds."""
    import numpy as np

    in_top = path == top_state
    first = int(np.argmax(in_top)) if in_top.any() else None
    return {
        "switches": int(np.count_nonzero(np.diff(path))),
        "first_top_idx": first,
        "top_frac": float(in_top.mean()),
    }

# ----------------------------- run it ---------------------------------------
import statistics as st
from collections import defaultdict
import numpy as np

DATA = '/content/drive/MyDrive/weirdspec/data'
STATES = 2
MIN_TRANSCRIPTS = 50   # fit only behaviors with at least this many transcripts
MIN_TOKENS = 20

meta = {i: m for i, m in enumerate(iter_jsonl(f'{DATA}/weird_meta.jsonl'))}
groups, rows_by_group = defaultdict(list), defaultdict(list)
for row in iter_jsonl(f'{DATA}/weird_scores.jsonl'):
    if len(row.get('nll_draft', [])) < MIN_TOKENS:
        continue
    behavior = meta[row['index']]['behavior_id']
    groups[behavior].append(excess_sequence(row))
    rows_by_group[behavior].append(row)
baseline = [excess_sequence(r) for r in iter_jsonl(f'{DATA}/baseline_scores.jsonl')
            if len(r.get('nll_draft', [])) >= MIN_TOKENS]
print(f'{sum(len(v) for v in groups.values())} weird sequences, {len(baseline)} baseline sequences')

lines = [
    '# Latent-state (HMM) tipping analysis', '',
    f'{STATES}-state Gaussian HMM over per-token excess; state 0 = normal mode,',
    'top state = tipped mode. `dwell` = expected run length 1/(1-a_ii);',
    '`switched` = share of transcripts that ever enter the tipped state;',
    '`first@` = median token index of first entry; `|hmm-peak|` = median distance',
    "between the HMM switch point and phase 5's argmax-excess token.", '',
    '| group | n_seq | mean_normal | mean_tipped | stay_norm | stay_tip | dwell_tip | tip% tokens | switched | first@ | \|hmm-peak\| |',
    '|---|---|---|---|---|---|---|---|---|---|---|',
]

def fit_and_report(name, seqs, rows):
    model = fit_gaussian_hmm(seqs, k=STATES)
    top = STATES - 1
    stats = [segment_stats(viterbi(x, model), top) for x in seqs]
    switched = [s for s in stats if s['first_top_idx'] is not None]
    dwell_t = 1.0 / max(1.0 - float(model['A'][top, top]), 1e-9)
    tip_frac = float(np.mean([s['top_frac'] for s in stats]))
    first_at = f"{st.median(s['first_top_idx'] for s in switched):.0f}" if switched else '-'
    peak_dist = '-'
    if rows is not None and switched:
        dists = [abs(s['first_top_idx'] - r['aggregates']['peak_excess_token_idx'])
                 for s, r in zip(stats, rows) if s['first_top_idx'] is not None]
        peak_dist = f'{st.median(dists):.0f}'
    lines.append(
        f"| {name} | {len(seqs)} | {model['means'][0]:.2f} | {model['means'][top]:.2f} "
        f"| {model['A'][0,0]:.3f} | {model['A'][top,top]:.3f} | {dwell_t:.0f} | {tip_frac:.0%} "
        f"| {len(switched)/len(stats):.0%} | {first_at} | {peak_dist} |")
    print('fitted:', name)

if baseline:
    fit_and_report('BASELINE (control)', baseline, None)
for behavior in sorted(groups, key=lambda b: -len(groups[b])):
    if len(groups[behavior]) >= MIN_TRANSCRIPTS:
        fit_and_report(behavior, groups[behavior], rows_by_group[behavior])

lines += ['', 'Reading guide: a real tipping behavior shows a well-separated tipped mean,',
          'a sticky tipped state (dwell >> 1) and a high switched share, while the',
          'BASELINE control should show little separation and near-zero tipped occupancy.']
report = '\n'.join(lines) + '\n'
open(f'{DATA}/hmm_report.md', 'w').write(report)
print(f'report written to {DATA}/hmm_report.md')

from IPython.display import Markdown, display
display(Markdown(report))